# 01. AGEGraph — Graph Mode

Apache AGE를 사용한 그래프 데이터 관리. Neo4j의 `Neo4jGraph`와 동일한 인터페이스.

**사전 준비**: Docker 컨테이너 실행
```bash
cd docker && docker compose up -d
```

In [ ]:
from langchain_age import AGEGraph

DSN = "host=localhost port=5433 dbname=langchain_age user=langchain password=langchain"

graph = AGEGraph(DSN, graph_name="demo_graph")

## 1. 노드와 관계 생성

표준 Cypher 문법을 그대로 사용한다. SQL 래핑은 라이브러리가 자동 처리.

In [ ]:
# 노드 생성
graph.query("MERGE (:Person {name: 'Alice', age: 30})")
graph.query("MERGE (:Person {name: 'Bob', age: 25})")
graph.query("MERGE (:Person {name: 'Charlie', age: 35})")

graph.query("MERGE (:Company {name: 'Acme Corp', industry: 'tech'})")
graph.query("MERGE (:Company {name: 'Globex', industry: 'finance'})")

# 관계 생성
graph.query(
    "MATCH (a:Person {name: 'Alice'}), (b:Person {name: 'Bob'}) "
    "MERGE (a)-[:KNOWS {since: 2020}]->(b)"
)
graph.query(
    "MATCH (a:Person {name: 'Alice'}), (c:Company {name: 'Acme Corp'}) "
    "MERGE (a)-[:WORKS_AT {role: 'engineer'}]->(c)"
)
graph.query(
    "MATCH (b:Person {name: 'Bob'}), (c:Company {name: 'Globex'}) "
    "MERGE (b)-[:WORKS_AT {role: 'analyst'}]->(c)"
)

print("Nodes and relationships created.")

## 2. 쿼리 실행

In [ ]:
# 기본 조회
results = graph.query("MATCH (n:Person) RETURN n.name AS name, n.age AS age")
for r in results:
    print(f"  {r['name']} (age: {r['age']})")

In [ ]:
# 관계 조회
results = graph.query(
    "MATCH (p:Person)-[r:WORKS_AT]->(c:Company) "
    "RETURN p.name AS person, r.role AS role, c.name AS company"
)
for r in results:
    print(f"  {r['person']} → {r['role']} at {r['company']}")

In [ ]:
# 집계
results = graph.query("MATCH (n:Person) RETURN count(n) AS total")
print(f"Total persons: {results[0]['total']}")

## 3. 스키마 조회

`refresh_schema()`를 호출하면 그래프의 노드 라벨, 관계 타입, 프로퍼티를 자동 탐지한다.

In [ ]:
graph.refresh_schema()
print(graph.schema)

In [ ]:
# structured_schema — dict 형태로 프로그래밍 활용
import json
print(json.dumps(graph.structured_schema, indent=2))

## 4. GraphDocument 일괄 추가

LLMGraphTransformer 등에서 생성한 `GraphDocument`를 그래프에 일괄 삽입.

In [ ]:
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

doc = GraphDocument(
    nodes=[
        Node(id="python", type="Language", properties={"paradigm": "multi"}),
        Node(id="django", type="Framework", properties={"stars": 75000}),
    ],
    relationships=[
        Relationship(
            source=Node(id="django", type="Framework"),
            target=Node(id="python", type="Language"),
            type="BUILT_WITH",
        ),
    ],
    source=Document(page_content="Django is a Python web framework."),
)

graph.add_graph_documents([doc], include_source=True)

# 확인
results = graph.query("MATCH (f:Framework)-[:BUILT_WITH]->(l:Language) RETURN f.id AS fw, l.id AS lang")
print(results)

## 5. Context Manager

`with` 문으로 연결을 자동 정리할 수 있다.

In [ ]:
with AGEGraph(DSN, "demo_graph", refresh_schema=False) as g:
    results = g.query("MATCH (n:Person) RETURN n.name AS name LIMIT 2")
    print(results)

# g._conn.closed == True (자동 정리됨)

## 6. 정리

In [ ]:
# 그래프 삭제 (주의: 되돌릴 수 없음)
# graph.drop_graph()
graph.close()
print("Done.")